In [33]:
import numpy as np 
import seaborn as sns 
import matplotlib.pyplot as plt 

import numpy as np 
import tensorflow as tf 
from tensorflow.keras import * 
from tensorflow.keras.layers import * 
from tensorflow.keras.models import load_model, save_model 
import tensorflow.keras.utils as utils
from keras.datasets import mnist 


### Original Floating-point Model 

In [48]:
def Net(): 
    # inputs = Input(shape=x_train.shape[1:])
    inputs = Input(shape=(28, 28, 1))  
    x = Conv2D(filters=4, kernel_size=(3,3), strides=(1,1), padding='same')(inputs)
    x = ReLU()(x)
    x = MaxPool2D((2,2),strides=(2,2), padding="same")(x)

    x = Flatten()(x)
    x = Dense(20)(x)

    x = ReLU()(x)
    x = Dense(10)(x)
    predictions = Softmax()(x)
    return Model(inputs=inputs, outputs=predictions)

In [49]:
epochs = 5 
num_classes = 10 
(x_train, y_train_num), (x_test, y_test_num) = mnist.load_data() 
x_train = x_train.reshape(x_train.shape[0], x_train.shape[1], x_train.shape[2], 1)
x_test = x_test.reshape(x_test.shape[0], x_test.shape[1], x_test.shape[2], 1)
y_train = utils.to_categorical(y_train_num, num_classes)
y_test = utils.to_categorical(y_test_num, num_classes)
print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)

model = Net()
model.load_weights('neural_network_model.h5')

(60000, 28, 28, 1) (10000, 28, 28, 1) (60000, 10) (10000, 10)


In [52]:
length = y_test.shape[0]
positives = 0
res = 0 
for i in range(length): 
    pred = model(x_test[i:i+1]).numpy().squeeze()
    y_test = (y_test > 0.5)
    pred = (pred > 0.5)
    # print("y_test :", y_test[i])
    # print("pred   :", pred)
    # print("res    :", y_test[i]==pred)
    positives = np.sum(y_test[i] == pred)
    if positives == 10: 
        res += 1

print("Accuracy:", res/length)

Accuracy: 0.885


In [53]:
"""
quantize numpy array
return:
q_x: quantized array
fp_x: reverse quantized array to float
"""
def quantize_array(x, bit_depth=8):
    min_x = x.min() 
    max_x = x.max()

    #find number of integer bits to represent this range
    int_bits = int(np.ceil(np.log2(max(abs(min_x),abs(max_x)))))
    
    if int_bits < 0: int_bits = 0

    frac_bits = bit_depth - 1 - int_bits #remaining bits are fractional bits (1-bit for sign)

    #floating point weights are scaled and rounded to [-2^(bit-1),2^(bit-1)-1], which are used in 
    #the fixed-point operations on the actual hardware (i.e., microcontroller)
    q_x = np.round(x*(2**frac_bits))

    #To quantify the impact of quantized weights, scale them back to
    # original range to run inference using quantized weights
    fp_x = q_x/(2**frac_bits)
    
    return q_x, fp_x, int_bits, frac_bits


"""
compute frac_bits based on max and min list
used for activation(feature map) quantization
"""
def min_max_quantize(max_list, min_list, bit_depth=8):
    frac_list = []
    for i in range(len(max_list)):
        int_bits = int(np.ceil(np.log2(max(abs(min_list[i]),abs(max_list[i])))))
        if int_bits < 0: int_bits = 0
        frac_bits = bit_depth - 1 - int_bits #remaining bits are fractional bits (1-bit for sign)
        frac_list.append(frac_bits)
    return frac_list

### Quantized Model

In [78]:
def QNet(bit=32, shift_list=[]):
    shift = False 
    if bit != 32: 
        shift = True 
    
    input = Input(shape=x_train.shape[1:], name='input')
    x = Conv2D(filters=4, kernel_size=(3,3), strides=(1,1), padding='same', name='conv')(input)
    if shift:
        x = x / 2**shift_list[0]
        x = tf.floor(x)
        x = tf.clip_by_value(x, -2**(bit-1), 2**(bit-1)-1)  
    x = ReLU()(x)

    x = MaxPool2D((2,2), strides=(2,2), padding='same', name='max_pool')(x)
    x = Flatten()(x)

    x = Dense(20, name='fc1')(x)
    if shift:
        x = x / 2**shift_list[1]
        x = tf.floor(x)
        x = tf.clip_by_value(x, -2**(bit-1), 2**(bit-1)-1)
    x = ReLU()(x)
    
    x = Dense(10, name='fc2')(x)
    if shift: 
        x = x / 2** shift_list[2]
        x = tf.floor(x)
        x = tf.clip_by_value(x, -2**(bit-1), 2**(bit-1)-1)
    if not shift:
        x_out = Softmax()(x)
    else:
        x_out = x
    
    return Model(input, x_out)

qmodel = QNet()
model.load_weights('neural_network_model.h5')

layer_names = [] 
for layer in model.layers: 
    if len(layer.get_weights()) != 0:
        layer_names.append(layer.name)
print("Layer names: ", layer_names)

Layer names:  ['conv2d_4', 'dense_8', 'dense_9']


In [76]:
layer_num = len(layer_names)
intermediate_output = [model.get_layer('input').output]
for i in range(layer_num):
    intermediate_output.append(model.get_layer(layer_names[i]).output)
    
intermediate_layer_model = Model(inputs = model.input, outputs = intermediate_output)

frac_bits_output_list = [] 
max_list = []
min_list = [] 
for i in range(length):
    pred = intermediate_layer_model(x_test[i:i+1])
    for i in range(len(pred)):
        pred_up = pred[i].numpy()
        max_np = pred_np.max()
        min_np = pred_np.min()
        if len(max_list) != len(pred):
            max_list.append(max_np)
            min_list.append(min_np)
        else:
            if max_np > max_list[i]:
                max_list[i] = max_np
            if min_np < min_list[i]:
                min_list[i] = min_np  
        
bit_depth = 16 
frac_bits_activation_list = min_max_quantize(max_list, min_list, bit_depth=bit_depth)

print("input: ", frac_bits_activation_list[0])
for i in range(layer_num):
    print(i+1, " ", layer_names[i], " ", frac_bits_activation_list[i+1])
    

ValueError: No such layer: input.